# Actividad 02. Proyecto en Python con Pandas

**Nombre:** David León Salas  
**Número de control:** 202680124  
**Materia:** Programación en Python y R

## 1. Datasets

Para esta actividad se utilizan los archivos de Spotify, Airbnb y viviendas de Melbourne proporcionados en el [directorio de datasets de la UCC](https://avh.ucc.mx/mod/folder/view.php?id=16476).

## 2. Carga de los archivos

En Jupyter local se usan los CSV de `src/Actividad01`. Se puede iniciar desde la raíz del proyecto, desde `src` o desde la carpeta de los CSV. Si un archivo no está localmente, se carga por su nombre desde el [repositorio](https://github.com/DvLeu/Introduccion_NumpyPandas), por lo que en ese caso se necesita internet.

En Colab se puede ejecutar el notebook sin subir los CSV: se leen de GitHub. Ejecutar las celdas de arriba hacia abajo con Pandas, NumPy y Matplotlib instalados. Las salidas se guardan en `datos_limpios` dentro de la carpeta de ejecución.


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 5)

# Primero se usan los archivos locales. Si no están, se leen de GitHub.
repositorio = "https://raw.githubusercontent.com/DvLeu/Introduccion_NumpyPandas/main/src/Actividad01/"

if Path("src/Actividad01").is_dir():
    carpeta = Path("src/Actividad01")
elif Path("Actividad01").is_dir():
    carpeta = Path("Actividad01")
else:
    carpeta = Path(".")

spotify_archivo = carpeta / "spotify_data clean.csv"
airbnb_archivo = carpeta / "Airbnb_Open_Data.csv"
melbourne_archivo = carpeta / "melb_data.csv"

spotify_fuente = spotify_archivo if spotify_archivo.is_file() else repositorio + "spotify_data%20clean.csv"
airbnb_fuente = airbnb_archivo if airbnb_archivo.is_file() else repositorio + "Airbnb_Open_Data.csv"
melbourne_fuente = melbourne_archivo if melbourne_archivo.is_file() else repositorio + "melb_data.csv"

spotify_original = pd.read_csv(spotify_fuente, dtype={"track_id": "string", "album_id": "string"})
airbnb_original = pd.read_csv(airbnb_fuente, dtype={"id": "string", "host id": "string"}, low_memory=False)
melbourne_original = pd.read_csv(melbourne_fuente)

salida = Path("datos_limpios")
salida.mkdir(exist_ok=True)

print("Filas de Spotify:", len(spotify_original))
print("Filas de Airbnb:", len(airbnb_original))
print("Filas de Melbourne:", len(melbourne_original))


HTTPError: HTTP Error 404: Not Found

## 3. Detección de problemas

### Spotify

In [ ]:
print("Filas y columnas:", spotify_original.shape)
display(spotify_original.head())
spotify_original.info()
print("Valores nulos por columna:")
display(spotify_original.isna().sum())
print("Filas duplicadas:", spotify_original.duplicated().sum())
display(spotify_original.describe().T)

### Airbnb

In [ ]:
print("Filas y columnas:", airbnb_original.shape)
display(airbnb_original.head())
airbnb_original.info()
print("Valores nulos por columna:")
display(airbnb_original.isna().sum())
print("Filas duplicadas:", airbnb_original.duplicated().sum())
display(airbnb_original.describe().T)

### Melbourne

In [ ]:
print("Filas y columnas:", melbourne_original.shape)
display(melbourne_original.head())
melbourne_original.info()
print("Valores nulos por columna:")
display(melbourne_original.isna().sum())
print("Filas duplicadas:", melbourne_original.duplicated().sum())
display(melbourne_original.describe().T)

### Revisión de valores incorrectos

In [ ]:
fechas_venta = pd.to_datetime(melbourne_original["Date"], format="%d/%m/%Y", errors="coerce")
problemas_detectados = pd.DataFrame([
    {"dataset": "Spotify", "problema": "Popularidad de canción fuera de 0 a 100", "cantidad": ((spotify_original["track_popularity"] < 0) | (spotify_original["track_popularity"] > 100)).sum()},
    {"dataset": "Spotify", "problema": "Duración no positiva", "cantidad": (spotify_original["track_duration_min"] <= 0).sum()},
    {"dataset": "Airbnb", "problema": "Disponibilidad fuera de 0 a 365", "cantidad": ((airbnb_original["availability 365"] < 0) | (airbnb_original["availability 365"] > 365)).sum()},
    {"dataset": "Airbnb", "problema": "Noches mínimas no positivas", "cantidad": (airbnb_original["minimum nights"] <= 0).sum()},
    {"dataset": "Airbnb", "problema": "Nombres de zonas mal escritos", "cantidad": airbnb_original["neighbourhood group"].isin(["brookln", "manhatan"]).sum()},
    {"dataset": "Melbourne", "problema": "Superficie construida no positiva", "cantidad": (melbourne_original["BuildingArea"] <= 0).sum()},
    {"dataset": "Melbourne", "problema": "Año anterior a 1800 o posterior a la venta", "cantidad": ((melbourne_original["YearBuilt"] < 1800) | (melbourne_original["YearBuilt"] > fechas_venta.dt.year)).sum()}
])
display(problemas_detectados)
print("Estos conteos incluyen las filas originales; los de limpieza se calculan después de quitar duplicados.")

## 4. Limpieza de los datos

Se eliminan filas repetidas, se revisan los tipos de datos y se corrigen valores incorrectos. Los datos que no se pueden recuperar se dejan vacíos. No se eliminan valores altos solo por ser poco frecuentes.

### Spotify

Los artistas y géneros faltantes se indican como "Sin dato". Se revisa que la popularidad esté entre 0 y 100, que la duración sea positiva y que las fechas se puedan convertir.

In [ ]:
spotify = spotify_original.copy()
spotify.columns = spotify.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for columna in spotify.select_dtypes(include=["object", "string"]).columns:
    spotify[columna] = spotify[columna].astype("string").str.strip().replace("", pd.NA)
spotify = spotify.drop_duplicates().reset_index(drop=True)
columnas_spotify = ["track_number", "track_popularity", "artist_popularity", "artist_followers", "album_total_tracks", "track_duration_min"]
for columna in columnas_spotify:
    spotify[columna] = pd.to_numeric(spotify[columna], errors="coerce")

problemas_spotify = {}
for columna in ["track_popularity", "artist_popularity"]:
    mascara = spotify[columna].notna() & ~spotify[columna].between(0, 100)
    problemas_spotify[columna] = int(mascara.sum())
    spotify.loc[mascara, columna] = np.nan

for columna, minimo in [("track_number", 1), ("artist_followers", 0), ("album_total_tracks", 1)]:
    mascara = (spotify[columna] < minimo) | (spotify[columna] % 1 != 0)
    problemas_spotify[columna] = int((mascara & spotify[columna].notna()).sum())
    spotify.loc[mascara, columna] = np.nan

problemas_spotify["duracion_no_positiva"] = int((spotify["track_duration_min"] <= 0).sum())
spotify.loc[spotify["track_duration_min"] <= 0, "track_duration_min"] = np.nan
fecha_original = spotify["album_release_date"].copy()
spotify["album_release_date"] = pd.to_datetime(fecha_original, format="%Y-%m-%d", errors="coerce")
problemas_spotify["fechas_no_interpretables"] = int((fecha_original.notna() & spotify["album_release_date"].isna()).sum())
for columna in ["artist_name", "artist_genres"]:
    spotify[columna + "_sin_dato"] = spotify[columna].isna()
    spotify[columna] = spotify[columna].fillna("Sin dato")
spotify["explicit"] = spotify["explicit"].astype("string").str.lower().map({"true": True, "false": False}).astype("boolean")
display(pd.Series(problemas_spotify, name="valores_incorrectos"))

### Airbnb

Se corrigen nombres de zonas y se convierten precios a números. La disponibilidad debe estar entre 0 y 365 y las noches mínimas deben ser positivas. Las reseñas por mes se completan con cero solamente cuando el total de reseñas es cero.

In [ ]:
airbnb = airbnb_original.copy()
airbnb.columns = airbnb.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for columna in airbnb.select_dtypes(include=["object", "string"]).columns:
    airbnb[columna] = airbnb[columna].astype("string").str.strip().replace("", pd.NA)
airbnb = airbnb.drop_duplicates().reset_index(drop=True)
airbnb["neighbourhood_group"] = airbnb["neighbourhood_group"].replace({"brookln": "Brooklyn", "manhatan": "Manhattan"})
for columna in ["price", "service_fee"]:
    texto = airbnb[columna].astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    airbnb[columna] = pd.to_numeric(texto, errors="coerce").astype(float)

columnas_airbnb = ["construction_year", "minimum_nights", "number_of_reviews", "reviews_per_month", "review_rate_number", "calculated_host_listings_count", "availability_365", "lat", "long"]
for columna in columnas_airbnb:
    airbnb[columna] = pd.to_numeric(airbnb[columna], errors="coerce")

reglas = {
    "price": airbnb["price"] <= 0,
    "service_fee": airbnb["service_fee"] < 0,
    "minimum_nights": (airbnb["minimum_nights"] < 1) | (airbnb["minimum_nights"] % 1 != 0),
    "number_of_reviews": (airbnb["number_of_reviews"] < 0) | (airbnb["number_of_reviews"] % 1 != 0),
    "reviews_per_month": airbnb["reviews_per_month"] < 0,
    "review_rate_number": ~airbnb["review_rate_number"].between(1, 5),
    "calculated_host_listings_count": (airbnb["calculated_host_listings_count"] < 1) | (airbnb["calculated_host_listings_count"] % 1 != 0),
    "availability_365": ~airbnb["availability_365"].between(0, 365) | (airbnb["availability_365"] % 1 != 0),
    "lat": ~airbnb["lat"].between(-90, 90),
    "long": ~airbnb["long"].between(-180, 180)
}
problemas_airbnb = {}
for columna, mascara in reglas.items():
    problemas_airbnb[columna] = int((mascara & airbnb[columna].notna()).sum())
    airbnb[columna + "_invalido"] = mascara & airbnb[columna].notna()
    airbnb.loc[mascara, columna] = np.nan

fecha_original = airbnb["last_review"].copy()
airbnb["last_review"] = pd.to_datetime(fecha_original, format="%m/%d/%Y", errors="coerce")
problemas_airbnb["fechas_no_interpretables"] = int((fecha_original.notna() & airbnb["last_review"].isna()).sum())
sin_resenas = airbnb["number_of_reviews"].eq(0) & airbnb["reviews_per_month"].isna()
airbnb["reviews_per_month_inferido"] = sin_resenas
airbnb.loc[sin_resenas, "reviews_per_month"] = 0
for columna in ["name", "host_name", "neighbourhood_group", "neighbourhood", "host_identity_verified", "cancellation_policy"]:
    airbnb[columna] = airbnb[columna].fillna("Sin dato")
airbnb["instant_bookable"] = airbnb["instant_bookable"].astype("string").str.lower().map({"true": True, "false": False}).astype("boolean")
display(pd.Series(problemas_airbnb, name="valores_incorrectos"))
print("Filas duplicadas eliminadas:", len(airbnb_original) - len(airbnb))
print("Identificadores repetidos restantes:", airbnb["id"].duplicated().sum())

### Melbourne

La superficie construida de cero se deja vacía. Los años anteriores a 1800 o posteriores a la venta se marcan para revisión y se dejan vacíos. Los estacionamientos faltantes se completan con la mediana por tipo de vivienda. Se conservan marcas para identificar estas modificaciones.

In [ ]:
melbourne = melbourne_original.copy()
melbourne.columns = melbourne.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
for columna in melbourne.select_dtypes(include=["object", "string"]).columns:
    melbourne[columna] = melbourne[columna].astype("string").str.strip().replace("", pd.NA)
melbourne = melbourne.drop_duplicates().reset_index(drop=True)
fecha_original = melbourne["date"].copy()
melbourne["date"] = pd.to_datetime(fecha_original, format="%d/%m/%Y", errors="coerce")
problemas_melbourne = {"fechas_no_interpretables": int((fecha_original.notna() & melbourne["date"].isna()).sum())}
columnas_melbourne = ["rooms", "price", "distance", "bedroom2", "bathroom", "car", "landsize", "buildingarea", "yearbuilt", "lattitude", "longtitude", "propertycount"]
for columna in columnas_melbourne:
    melbourne[columna] = pd.to_numeric(melbourne[columna], errors="coerce")
for columna in ["rooms", "price", "buildingarea"]:
    mascara = melbourne[columna] <= 0
    problemas_melbourne[columna] = int(mascara.sum())
    melbourne.loc[mascara, columna] = np.nan
for columna in ["distance", "bedroom2", "bathroom", "car", "landsize", "propertycount"]:
    mascara = melbourne[columna] < 0
    problemas_melbourne[columna] = int(mascara.sum())
    melbourne.loc[mascara, columna] = np.nan

anio_venta = melbourne["date"].dt.year
anio_invalido = (melbourne["yearbuilt"] < 1800) | (melbourne["yearbuilt"] > anio_venta) | (melbourne["yearbuilt"].notna() & (melbourne["yearbuilt"] % 1 != 0))
melbourne["yearbuilt_invalido"] = anio_invalido
problemas_melbourne["yearbuilt"] = int(anio_invalido.sum())
melbourne.loc[anio_invalido, "yearbuilt"] = np.nan
melbourne["car_imputado"] = melbourne["car"].isna()
mediana_por_tipo = melbourne.groupby("type")["car"].transform("median")
melbourne["car"] = melbourne["car"].fillna(mediana_por_tipo).fillna(melbourne["car"].median())
melbourne["councilarea"] = melbourne["councilarea"].fillna("Sin dato")
melbourne["postcode"] = pd.to_numeric(melbourne["postcode"], errors="coerce").astype("Int64").astype("string")
display(pd.Series(problemas_melbourne, name="valores_incorrectos"))
print("Estacionamientos imputados:", melbourne["car_imputado"].sum())

## 5. Nuevas columnas y uso de memoria

### Spotify

Se agrega el año de lanzamiento, la duración en segundos y el nivel de popularidad. Para este ejercicio, baja corresponde a 0–33, media a 34–66 y alta a 67–100.

In [ ]:
spotify["anio_lanzamiento"] = spotify["album_release_date"].dt.year.astype("Int64")
spotify["duracion_segundos"] = spotify["track_duration_min"] * 60
spotify["nivel_popularidad"] = pd.cut(spotify["track_popularity"], bins=[-1, 33, 66, 100], labels=["Baja", "Media", "Alta"])
memoria_antes = spotify.memory_usage(deep=True).sum()
for columna in ['album_type', 'artist_name']:
    spotify[columna] = spotify[columna].astype("category")
memoria_despues = spotify.memory_usage(deep=True).sum()
print(f"Memoria antes: {memoria_antes / 1024**2:.2f} MB")
print(f"Memoria después: {memoria_despues / 1024**2:.2f} MB")
display(spotify.groupby("album_type", observed=True).agg(
    canciones=("track_id", "count"),
    popularidad_media=("track_popularity", "mean"),
    duracion_media_min=("track_duration_min", "mean")
).round(2))

### Airbnb

Se calcula la proporción de días disponibles y se identifican estancias mínimas mayores de un año. La disponibilidad no equivale a ocupación.

In [ ]:
airbnb["proporcion_disponibilidad"] = airbnb["availability_365"] / 365
# Booleano nullable: una estancia desconocida permanece desconocida.
airbnb["estancia_minima_mayor_un_anio"] = airbnb["minimum_nights"].astype("Float64") > 365
memoria_antes = airbnb.memory_usage(deep=True).sum()
for columna in ['neighbourhood_group', 'room_type', 'cancellation_policy', 'host_identity_verified']:
    airbnb[columna] = airbnb[columna].astype("category")
memoria_despues = airbnb.memory_usage(deep=True).sum()
print(f"Memoria antes: {memoria_antes / 1024**2:.2f} MB")
print(f"Memoria después: {memoria_despues / 1024**2:.2f} MB")
display(airbnb.groupby("neighbourhood_group", observed=True).agg(
    anuncios=("id", "count"),
    precios_validos=("price", "count"),
    precio_mediano=("price", "median"),
    disponibilidad_media=("availability_365", "mean")
).round(2))

### Melbourne

Se calcula la antigüedad al vender y el precio por unidad de superficie construida. Los precios fuera del rango intercuartílico ampliado se marcan como atípicos, pero se conservan.

In [ ]:
melbourne["anio_venta"] = melbourne["date"].dt.year.astype("Int64")
melbourne["antiguedad_venta"] = melbourne["anio_venta"] - melbourne["yearbuilt"]
melbourne["precio_por_superficie"] = melbourne["price"] / melbourne["buildingarea"]
q1 = melbourne["price"].quantile(0.25)
q3 = melbourne["price"].quantile(0.75)
ric = q3 - q1
melbourne["precio_atipico"] = (melbourne["price"] < q1 - 1.5 * ric) | (melbourne["price"] > q3 + 1.5 * ric)
memoria_antes = melbourne.memory_usage(deep=True).sum()
for columna in ['type', 'method', 'regionname', 'councilarea']:
    melbourne[columna] = melbourne[columna].astype("category")
memoria_despues = melbourne.memory_usage(deep=True).sum()
print(f"Memoria antes: {memoria_antes / 1024**2:.2f} MB")
print(f"Memoria después: {memoria_despues / 1024**2:.2f} MB")
display(melbourne.groupby("type", observed=True).agg(
    operaciones=("address", "count"),
    precio_mediano=("price", "median"),
    superficie_mediana=("buildingarea", "median")
).round(2))
print("Precios atípicos conservados:", melbourne["precio_atipico"].sum())

## 6. Visualizaciones

Las correlaciones se calculan con los datos disponibles de cada pareja de columnas. No indican una relación de causa y efecto. Para los scatter se usa una muestra de hasta 5,000 filas con una semilla fija.

### Spotify: correlaciones y mapa de calor

In [ ]:
columnas = ['track_popularity', 'artist_popularity', 'artist_followers', 'track_duration_min']
correlacion = spotify[columnas].corr()
display(correlacion.round(3))

plt.figure(figsize=(8, 6))
plt.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(columnas)), columnas, rotation=45, ha="right")
plt.yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        plt.text(j, i, f"{correlacion.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Spotify: mapa de calor")
plt.tight_layout()
plt.show()

### Spotify: boxplot

In [ ]:
spotify.boxplot(column="track_duration_min", figsize=(8, 5))
plt.title("Spotify: distribución de track_duration_min")
plt.ylabel("track_duration_min")
plt.tight_layout()
plt.show()

### Spotify: scatter

In [ ]:
pares = spotify[["artist_popularity", "track_popularity"]].dropna()
muestra = pares.sample(n=min(5000, len(pares)), random_state=42)
muestra.plot.scatter(x="artist_popularity", y="track_popularity", alpha=0.25, s=12, figsize=(8, 5))
plt.title("Spotify: artist_popularity y track_popularity")
plt.tight_layout()
plt.show()

### Airbnb: correlaciones y mapa de calor

In [ ]:
columnas = ['price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'availability_365']
correlacion = airbnb[columnas].corr()
display(correlacion.round(3))

plt.figure(figsize=(8, 6))
plt.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(columnas)), columnas, rotation=45, ha="right")
plt.yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        plt.text(j, i, f"{correlacion.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Airbnb: mapa de calor")
plt.tight_layout()
plt.show()

### Airbnb: boxplot

In [ ]:
airbnb.boxplot(column="price", figsize=(8, 5))
plt.title("Airbnb: distribución de price")
plt.ylabel("price")
plt.tight_layout()
plt.show()

### Airbnb: scatter

In [ ]:
pares = airbnb[["price", "availability_365"]].dropna()
muestra = pares.sample(n=min(5000, len(pares)), random_state=42)
muestra.plot.scatter(x="price", y="availability_365", alpha=0.25, s=12, figsize=(8, 5))
plt.title("Airbnb: price y availability_365")
plt.tight_layout()
plt.show()

### Melbourne: correlaciones y mapa de calor

In [ ]:
columnas = ['price', 'rooms', 'distance', 'buildingarea', 'antiguedad_venta']
correlacion = melbourne[columnas].corr()
display(correlacion.round(3))

plt.figure(figsize=(8, 6))
plt.imshow(correlacion, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(columnas)), columnas, rotation=45, ha="right")
plt.yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        plt.text(j, i, f"{correlacion.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Melbourne: mapa de calor")
plt.tight_layout()
plt.show()

### Melbourne: boxplot

In [ ]:
melbourne.boxplot(column="price", figsize=(8, 5))
plt.title("Melbourne: distribución de price")
plt.ylabel("price")
plt.tight_layout()
plt.show()

### Melbourne: scatter

In [ ]:
pares = melbourne[["distance", "price"]].dropna()
muestra = pares.sample(n=min(5000, len(pares)), random_state=42)
muestra.plot.scatter(x="distance", y="price", alpha=0.25, s=12, figsize=(8, 5))
plt.title("Melbourne: distance y price")
plt.tight_layout()
plt.show()

### Resultados

En Spotify hay una relación positiva moderada entre la popularidad del artista y la canción. En Airbnb, el precio y la tarifa de servicio están muy relacionados, mientras que el precio y la disponibilidad no muestran una relación lineal clara. En Melbourne, la relación entre distancia y precio es débil y negativa.

Los géneros faltantes de Spotify y las superficies desconocidas de Melbourne limitan las comparaciones que se pueden hacer con esas columnas.

In [ ]:
print(f"Spotify: {len(spotify):,} canciones; {spotify['artist_genres_sin_dato'].sum():,} sin género informado.")
print(f"Correlación popularidad del artista/canción: {spotify['artist_popularity'].corr(spotify['track_popularity']):.3f}.")
print(f"Airbnb: {len(airbnb):,} anuncios; se eliminaron {len(airbnb_original) - len(airbnb):,} duplicados.")
print(f"Disponibilidades fuera de rango detectadas: {problemas_airbnb['availability_365']:,}.")
print(f"Melbourne: {len(melbourne):,} operaciones; {melbourne['buildingarea'].isna().mean():.1%} sin superficie construida válida.")
print(f"Correlación distancia/precio: {melbourne['distance'].corr(melbourne['price']):.3f}.")

## 7. Guardar los datasets limpios

Los archivos se guardan en `datos_limpios`, sin agregar el índice. Los CSV conservan los valores, pero al cargarlos otra vez se deben revisar los tipos de fechas y categorías. En Colab hay que descargar las salidas antes de cerrar la sesión.

In [ ]:
spotify.to_csv(salida / "spotify_limpio.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
airbnb.to_csv(salida / "airbnb_limpio.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
melbourne.to_csv(salida / "melbourne_limpio.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")

resumen = pd.DataFrame({
    "dataset": ["Spotify", "Airbnb", "Melbourne"],
    "filas_originales": [len(spotify_original), len(airbnb_original), len(melbourne_original)],
    "filas_limpias": [len(spotify), len(airbnb), len(melbourne)]
})
resumen["filas_eliminadas"] = resumen["filas_originales"] - resumen["filas_limpias"]
display(resumen)

print("Nulos restantes en Spotify:")
display(spotify.isna().sum())
print("Nulos restantes en Airbnb:")
display(airbnb.isna().sum())
print("Nulos restantes en Melbourne:")
display(melbourne.isna().sum())
print("Archivos guardados en:", salida)